In [1]:
from pathlib import Path

"""
Replace with your desired directories
"""

from utility import repo_root

# FIG_DIR = Path("/Users/zach/2025_RA/matthias/final images/4_expanded_sleep")
BIDS_DIR = Path("/Users/zach/2025_RA/matthias/bids_sleepmreye")
CONFIG_DIR = Path(repo_root() / "sleepmreye" / "designs")

In [2]:
""" Load data """

from mreyemove.analysis.glm.cli import construct_desc
from mreyemove.analysis.glm.config import GLMConfig
from SleepMReye.sleepmreye.sleep_mrio import SleepMRIO

mrio = SleepMRIO(
    bids_root=BIDS_DIR,
    tr=2.1,
    task="sleep",
    load_signal=True,
    desc_add="clean",
    ignore_boundary_trs=0,
    load_eog=False,
)


In [8]:
import numpy as np
from nilearn.surface import vol_to_surf, load_surf_mesh
from nilearn.image import new_img_like

from mreyemove.analysis.searchlight_correlation import geodesic_neighborhoods


# --------------------------------- helpers -----------------------------------
def jaccard(a, b):
    if not a and not b:
        return np.nan
    return len(a & b) / len(a | b)


def sphere_mask(r):
    """Boolean (2r+1)^3 cube, True where index distance <= r."""
    z, y, x = np.mgrid[-r:r + 1, -r:r + 1, -r:r + 1]
    return (x * x + y * y + z * z) <= r * r


def volumetric_vset(center_world, vol, vol_data, sph, r, project_fn, threshold):
    """Place a voxel sphere under `center_world`, project to the surface, and
    return the set of vertices it covers."""
    # Use affine to find the voxel (rounded) corresponding to some vertex in MNI space 
    ijk = np.linalg.inv(vol.affine) @ np.r_[center_world, 1.0]
    cx, cy, cz = np.round(ijk[:3]).astype(int)

    # Ignore if center makes sphere fall outside of volume 
    s = vol_data.shape
    if not (r <= cx < s[0] - r and r <= cy < s[1] - r and r <= cz < s[2] - r):
        return set()  # box leaves FOV -> dropped downstream

    # Create a mask on the volume and then project to the surface
    # Using the passed in projection function
    nb = np.zeros(s, dtype=np.float32)
    nb[cx - r:cx + r + 1, cy - r:cy + r + 1, cz - r:cz + r + 1][sph] = 1
    surf_vals = project_fn(new_img_like(vol, nb))            # (V,)
    return set(np.where(surf_vals > threshold)[0])


# ------------------------------- core routine --------------------------------
def calibrate_searchlight_radius(
    coords_geodesic, faces, coords_world, vol, project_fn,
    sphere_radius_vox, radii_mm,
    n_centers=1000, rng_seed=0, proj_threshold=0.5, label="",
):
    n_vertices = coords_geodesic.shape[0]
    assert coords_world.shape[0] == n_vertices, "coords_world / mesh vertex count mismatch"

    vol_data = vol.get_fdata()
    sph = sphere_mask(sphere_radius_vox)    
    rng = np.random.default_rng(rng_seed)
    centers = rng.choice(n_vertices, size=n_centers, replace=False)

    # Get the set of vertices contained in the projected volumetric searchlight
    vset_by_center = {
        c: volumetric_vset(coords_world[c], vol, vol_data, sph,
                           sphere_radius_vox, project_fn, proj_threshold)
        for c in centers
    }
    # Drop any searchlights where the mask extended outside of the volume
    usable = [int(c) for c in centers if vset_by_center[c]]
    dropped = len(centers) - len(usable)
    if dropped:
        print(f"[{label}] dropped {dropped} centre(s) with empty projection.")

    # For every radius get the jaccard overlap with the actual surface neighbourhood
    # for each usable center vertex and take the median jaccard across all centers 
    median_jaccard_by_radius = []
    for radius_mm in radii_mm:
        nbrs = geodesic_neighborhoods(coords_geodesic, faces, radius_mm=radius_mm)
        jaccard_by_center = {
            c: jaccard(vset_by_center[c], set(nbrs[c])) for c in usable
        }
        median_jaccard = np.median(list(jaccard_by_center.values()))
        median_jaccard_by_radius.append(median_jaccard)
        print(f"radius: {radius_mm}: median jaccard: {median_jaccard}")

    best_idx = int(np.nanargmax(list(median_jaccard_by_radius)))
    if best_idx in (0, len(radii_mm) - 1):
        print(f"[{label}] warning: best radius at sweep edge -> widen radii_mm range")
        print(f"[{label}] best radius: {radii_mm[best_idx]:.1f} mm "
              f"(median Jaccard {median_jaccard_by_radius[best_idx]:.3f})")
    return float(radii_mm[best_idx]), float(median_jaccard_by_radius[best_idx])


In [4]:
config = GLMConfig.from_yaml(
    CONFIG_DIR / "original_sleep.yml"
)
desc_add = construct_desc(dataset=mrio, config=config)
desc_add = f"{desc_add}-{config.second_level.method}"
data_a, _ = mrio.load_FLAME_result(contrast="mreyemove_W", desc_add=desc_add)
vol = data_a.t_stat

In [11]:
def run_cortex(vol, sphere_radius_vox=3, radii_mm=np.arange(8.0, 22.0, 1.0), n_centers=1000):
    from nilearn import datasets
    fs = datasets.fetch_surf_fsaverage(mesh="fsaverage7")
    coords_pial, faces = load_surf_mesh(fs["pial_left"])
    coords_mid = coords_pial

    def project(nb_img):
        return vol_to_surf(nb_img, fs["pial_left"])

    return calibrate_searchlight_radius(
        coords_geodesic=coords_mid, faces=faces, coords_world=coords_mid,
        vol=vol, project_fn=project,
        sphere_radius_vox=sphere_radius_vox, radii_mm=radii_mm, label="cortex", n_centers=n_centers,
    )



run_cortex(vol, sphere_radius_vox=3, radii_mm=np.arange(19.0, 25.0, 1.0), n_centers=10000)

[fetch_surf_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Building geodesic neighborhoods (radius=19.0mm)...


Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 571, Median: 536, Min: 175, Max: 1438
radius: 19.0: median jaccard: 0.5716504120978257
Building geodesic neighborhoods (radius=20.0mm)...


Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 631, Median: 594, Min: 189, Max: 1577
radius: 20.0: median jaccard: 0.611290606294202
Building geodesic neighborhoods (radius=21.0mm)...


...

Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 830, Median: 787, Min: 254, Max: 2009
radius: 23.0: median jaccard: 0.6749836173001311
Building geodesic neighborhoods (radius=24.0mm)...


Dijkstra neighborhoods:   0%|          | 0/328 [00:00<?, ?it/s]

Neighbors — Mean: 902, Median: 858, Min: 281, Max: 2157
radius: 24.0: median jaccard: 0.6747177658942365


(23.0, 0.6749836173001311)

In [10]:
def run_cerebellum(vol, sphere_radius_vox=3, radii_mm=np.arange(4.0, 20.0, 1.0), n_centers=1000):
    import os
    import nibabel as nib
    import SUITPy
    from SUITPy import flatmap as suit_flatmap

    surf_dir = os.path.join(os.path.dirname(SUITPy.__file__), "surfaces")
    surf = nib.load(os.path.join(surf_dir, "PIAL_FSL.surf.gii"))
    verts = surf.darrays[0].data               
    faces = surf.darrays[1].data

    def project(nb_img):
        return np.asarray(suit_flatmap.vol_to_surf([nb_img], space="FSL")).ravel()

    return calibrate_searchlight_radius(
        coords_geodesic=verts, faces=faces, coords_world=verts,
        vol=vol, project_fn=project,
        sphere_radius_vox=sphere_radius_vox, radii_mm=radii_mm,
        label=f"cerebellum", n_centers=n_centers,
    )


run_cerebellum(vol, sphere_radius_vox=2, radii_mm=np.arange(10.0, 15.0, 1.0),  n_centers=10000)


[cerebellum] dropped 1965 centre(s) with empty projection.
Building geodesic neighborhoods (radius=10.0mm)...


Dijkstra neighborhoods:   0%|          | 0/58 [00:00<?, ?it/s]

Neighbors — Mean: 103, Median: 103, Min: 1, Max: 160
radius: 10.0: median jaccard: 0.21390374331550802
Building geodesic neighborhoods (radius=11.0mm)...


Dijkstra neighborhoods:   0%|          | 0/58 [00:00<?, ?it/s]

Neighbors — Mean: 124, Median: 125, Min: 1, Max: 190
radius: 11.0: median jaccard: 0.21875
Building geodesic neighborhoods (radius=12.0mm)...


...

Dijkstra neighborhoods:   0%|          | 0/58 [00:00<?, ?it/s]

Neighbors — Mean: 173, Median: 174, Min: 1, Max: 258
radius: 13.0: median jaccard: 0.2102803738317757
Building geodesic neighborhoods (radius=14.0mm)...


Dijkstra neighborhoods:   0%|          | 0/58 [00:00<?, ?it/s]

Neighbors — Mean: 200, Median: 202, Min: 1, Max: 295
radius: 14.0: median jaccard: 0.2019704433497537


(11.0, 0.21875)